# Seeds: how much of each result is run-to-run noise?

Every training so far used seed 0. The confidence intervals resample conversations, so they cover the variation of the test set but not of training (LoRA initialisation and data order). Here the final recipe (decision 15) and the data curve (decision 18) are trained with seeds 0, 1 and 2.

All checkpoints are picked on the 20-turn validation set, so the three data sizes are comparable (in decision 18 the 100% point was picked on the 6-turn one). Seed 0 of the 100% run is retrained, because only its step 2,500 was kept and it was picked on the 6-turn set; 25% and 50% with seed 0 already exist with this protocol (`checkpoints_data25.json`, `checkpoints_data50.json`). Every training runs on the same Mac with the same recipe as the published results. The evaluations ran on the NAS GPU (TLM_PY set), which matches the Mac to 3e-4 nats on the same checkpoints (calib-cuda3: data25 steps 300 and 1400, 3.53912 vs 3.53915 and 4.10193 vs 4.10159).

## Prediction (written before running)

This is more a check than a hypothesis. I expect the experiment to do what it's meant to: show that the results are consistent across seeds and not a product of chance. Three seeds seem representative enough for an experiment this small.

In [1]:
# Each heavy step runs a script from scripts/ through run(), which stops the notebook on failure. They all resume: re-running this notebook skips
# what is already done (trainings with a final adapters.safetensors, generations already written,
# evaluations already in their JSON). Run headless with:
#   cd experiments && nohup uv run jupyter nbconvert --to notebook --execute --inplace \
#       --ExecutePreprocessor.timeout=-1 Baselines.ipynb > Baselines.log 2>&1 &
import json, os, subprocess
from pathlib import Path
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
# On the Mac: "uv run python". On the NAS container (CUDA): TLM_PY=python, the image's venv with mlx-cuda.
PY = os.environ.get("TLM_PY", "uv run python")
# On the NAS (10 GB of VRAM) TLM_TRAIN_EXTRA="--grad-accum 2": micro-batches of 2, same effective batch of 4.
TRAIN_EXTRA = os.environ.get("TLM_TRAIN_EXTRA", "")
ROOT = Path("..").resolve()
R = ROOT / "experiments" / "results"
M05 = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
M15 = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"
M3 = "mlx-community/Qwen2.5-3B-Instruct-4bit"
JUDGE = "mlx-community/Qwen2.5-7B-Instruct-4bit"  # generates and trains nothing here
V6 = "data/dataset/session/valid.jsonl"
V20 = "data/dataset/context/valid_ctx20.jsonl"
TURNS_V1 = "data/processed_v1_2026-10-17/turns.jsonl"  # the turns session/ was built from

def run(cmd):
    """Run a command from the repo root, show its output here, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Fetching" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

def done_training(name, iters):
    # The last numbered checkpoint, not adapters.safetensors: mlx-lm rewrites that one at every save,
    # so it also exists after a training that crashed halfway.
    return (ROOT / "adapters" / name / f"{iters:07d}_adapters.safetensors").exists()

def best_step(path):
    return json.load(open(path))["best"]["step"]

## 1. The final recipe, 100% of the data, seeds 0 to 2

In [2]:
for seed in [0, 1, 2]:
    name = f"run1_seed{seed}"
    if not done_training(name, 3500):
        run(f"{PY} scripts/train.py --name {name} --data data/dataset/session --seed {seed} --iters 3500 {TRAIN_EXTRA}")
    run(f"{PY} scripts/eval_checkpoints.py adapters/{name} --data {V20} --steps 1000:3500:500 --out experiments/results/checkpoints_{name}.json")

$ /opt/venv/bin/python scripts/eval_checkpoints.py adapters/run1_seed0 --data data/dataset/context/valid_ctx20.jsonl --steps 1000:3500:500 --out experiments/results/checkpoints_run1_seed0.json
step  1000: loss 3.425
step  1500: loss 3.417
step  2000: loss 3.403
step  2500: loss 3.394
step  3000: loss 3.436
step  3500: loss 3.443
best: step 2500, loss 3.394
$ /opt/venv/bin/python scripts/eval_checkpoints.py adapters/run1_seed1 --data data/dataset/context/valid_ctx20.jsonl --steps 1000:3500:500 --out experiments/results/checkpoints_run1_seed1.json
step  1000: loss 3.432
step  1500: loss 3.411
step  2000: loss 3.412
step  2500: loss 3.387
step  3000: loss 3.437
step  3500: loss 3.418
best: step 2500, loss 3.387
$ /opt/venv/bin/python scripts/eval_checkpoints.py adapters/run1_seed2 --data data/dataset/context/valid_ctx20.jsonl --steps 1000:3500:500 --out experiments/results/checkpoints_run1_seed2.json
step  1000: loss 3.432
step  1500: loss 3.412
step  2000: loss 3.407
step  2500: loss 3.3

In [3]:
for seed in [0, 1, 2]:
    name = f"run1_seed{seed}"
    S = best_step(R / f"checkpoints_{name}.json")
    run(f"{PY} scripts/eval_signal.py --adapter adapters/{name} --step {S} --turns {TURNS_V1} --out experiments/results/signal_{name}.json")

$ /opt/venv/bin/python scripts/eval_signal.py --adapter adapters/run1_seed0 --step 2500 --turns data/processed_v1_2026-10-17/turns.jsonl --out experiments/results/signal_run1_seed0.json
mine:    base 75.5 -> adapter 29.6 (60.8% drop)
friends: base 81.1 -> adapter 47.9 (41.0% drop)
signal: 19.8 points, 95% CI [14.7, 25.8]
$ /opt/venv/bin/python scripts/eval_signal.py --adapter adapters/run1_seed1 --step 2500 --turns data/processed_v1_2026-10-17/turns.jsonl --out experiments/results/signal_run1_seed1.json
mine:    base 75.5 -> adapter 29.4 (61.0% drop)
friends: base 81.1 -> adapter 46.6 (42.6% drop)
signal: 18.4 points, 95% CI [13.3, 24.1]
$ /opt/venv/bin/python scripts/eval_signal.py --adapter adapters/run1_seed2 --step 2500 --turns data/processed_v1_2026-10-17/turns.jsonl --out experiments/results/signal_run1_seed2.json
mine:    base 75.5 -> adapter 29.1 (61.4% drop)
friends: base 81.1 -> adapter 47.1 (41.9% drop)
signal: 19.6 points, 95% CI [14.2, 25.5]


## 2. The data curve, 25% and 50%, seeds 1 and 2

In [4]:
for seed in [1, 2]:
    for pct, iters, steps in [(25, 1400, "100:1400:100"), (50, 2400, "200:2400:200")]:
        name = f"data{pct}_seed{seed}"
        if not done_training(name, iters):
            run(f"{PY} scripts/train.py --name {name} --data data/dataset/session_{pct} --seed {seed} --iters {iters} {TRAIN_EXTRA}")
        run(f"{PY} scripts/eval_checkpoints.py adapters/{name} --data {V20} --steps {steps} --out experiments/results/checkpoints_{name}.json")

$ /opt/venv/bin/python scripts/eval_checkpoints.py adapters/data25_seed1 --data data/dataset/context/valid_ctx20.jsonl --steps 100:1400:100 --out experiments/results/checkpoints_data25_seed1.json
step   100: loss 3.639
step   200: loss 3.575
step   300: loss 3.542
step   400: loss 3.554
step   500: loss 3.567
step   600: loss 3.545
step   700: loss 3.603
step   800: loss 3.660
step   900: loss 3.642
step  1000: loss 3.676
step  1100: loss 3.813
step  1200: loss 3.829
step  1300: loss 3.846
step  1400: loss 4.070
best: step 300, loss 3.542
$ /opt/venv/bin/python scripts/eval_checkpoints.py adapters/data50_seed1 --data data/dataset/context/valid_ctx20.jsonl --steps 200:2400:200 --out experiments/results/checkpoints_data50_seed1.json
step   200: loss 3.593
step   400: loss 3.511
step   600: loss 3.484
step   800: loss 3.481
step  1000: loss 3.494
step  1200: loss 3.467
step  1400: loss 3.518
step  1600: loss 3.547
step  1800: loss 3.526
step  2000: loss 3.545
step  2200: loss 3.647
step  

## Summary

In [5]:
import numpy as np

def best(path):
    d = json.load(open(path))
    v = d.get("valid") or d.get("valid_run1_ctx20")
    v = {int(k): x for k, x in v.items()}
    s = min(v, key=v.get)
    return s, v[s]

print("best validation loss (20 turns), by seed")
table = {}
for pct in [25, 50, 100]:
    vals = []
    for seed in [0, 1, 2]:
        if pct == 100:
            path = R / f"checkpoints_run1_seed{seed}.json"
        else:
            path = R / (f"checkpoints_data{pct}.json" if seed == 0 else f"checkpoints_data{pct}_seed{seed}.json")
        step, loss = best(path)
        vals.append(loss)
        print(f"  {pct:>3}% seed {seed}: {loss:.3f} at step {step}")
    table[pct] = vals
    print(f"  {pct:>3}%: mean {np.mean(vals):.3f}, sd {np.std(vals, ddof=1):.3f}")

print("\nsignal (me - friends, percentage points), by seed")
sig = []
for seed in [0, 1, 2]:
    d = json.load(open(R / f"signal_run1_seed{seed}.json"))
    sig.append(d["signal"] * 100)
    print(f"  seed {seed}: {d['signal'] * 100:.1f} [{d['signal_ci'][0] * 100:.1f}, {d['signal_ci'][1] * 100:.1f}]")
print(f"  mean {np.mean(sig):.1f}, sd {np.std(sig, ddof=1):.1f}")

best validation loss (20 turns), by seed
   25% seed 0: 3.539 at step 300
   25% seed 1: 3.542 at step 300
   25% seed 2: 3.541 at step 300
   25%: mean 3.541, sd 0.001
   50% seed 0: 3.474 at step 1200
   50% seed 1: 3.467 at step 1200
   50% seed 2: 3.469 at step 1200
   50%: mean 3.470, sd 0.004
  100% seed 0: 3.394 at step 2500
  100% seed 1: 3.387 at step 2500
  100% seed 2: 3.386 at step 2500
  100%: mean 3.389, sd 0.004
signal (me - friends, percentage points), by seed
  seed 0: 19.8 [14.7, 25.8]
  seed 1: 18.4 [13.3, 24.1]
  seed 2: 19.6 [14.2, 25.5]
  mean 19.3, sd 0.7


## Conclusion

**Prediction:** a check more than a hypothesis: the results hold across seeds, and three seeds are enough.

**Confirmed, with less noise than expected.**
- **Final recipe (100%):** best validation loss 3.394 / 3.387 / 3.386, all three at step 2,500 (sd 0.004). The published step 2,500 is not a lucky pick.
- **Me-vs-friends signal:** 19.8 / 18.4 / 19.6 points (sd 0.7), against confidence intervals of about ±5.5 per seed. Training noise is small next to the test-set noise; the main result of the project doesn't depend on the seed.
- **Data curve:** 25% 3.541 (sd 0.001), 50% 3.470 (sd 0.004), 100% 3.389 (sd 0.004). Each doubling lowers the loss by 0.07-0.08 nats, about 20 times the seed spread, and every seed picks the same step at each size (300, 1,200, 2,500). Decision 18, "more data still lowers the loss", holds.

**Why so stable:** the seeds do change the training (the LoRA weights of seeds 1 and 2 differ already at step 100), but with lr 1e-5 and batch 4 the runs end up in the same place. The NAS recipe (micro-batch 1, 4 accumulation steps, gradient checkpointing; AllLayersSeeds.ipynb) spread more, up to 0.023 (sd ~0.012). With three seeds per side it's not clear whether that difference is real; if it is, it comes from the recipe, not the machine, since evaluation matches across them.

**Rule for the rest of the project:** on the Mac recipe, differences between single runs below ~0.01 nats aren't real; on the NAS recipe, below ~0.02-0.025.
